In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import joblib

os.makedirs('../models', exist_ok=True)

# =========================
# Load Dataset
# =========================
df = pd.read_csv('../data/pipeline_dataset.csv')

X = df.drop(columns=['sensor_id', 'leak_position', 'leak_size']).values
Y_pos = df['leak_position'].values.astype(np.float32)
Y_size = df['leak_size'].values

# تبدیل کلاس‌ها به 0 ... n-1
Y_size = Y_size - Y_size.min()
num_classes = len(np.unique(Y_size))

print("Dataset Shape:", X.shape)

# =========================
# Train/Test Split
# =========================
X_train, X_test, y_pos_train, y_pos_test, y_size_train, y_size_test = train_test_split(
    X,
    Y_pos,
    Y_size,
    test_size=0.2,
    random_state=42
)

# =========================
# Standardization
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, "../models/acoustic_scaler.pkl")

input_length = X_train_scaled.shape[1]

print("Input Length:", input_length)


Dataset Shape: (5000, 23)
Input Length: 23


In [2]:
# =========================
# Dataset Class
# =========================
class AcousticDataset(Dataset):

    def __init__(self, x, y_pos, y_size):

        self.x = torch.tensor(x, dtype=torch.float32)
        self.y_pos = torch.tensor(y_pos, dtype=torch.float32)
        self.y_size = torch.tensor(y_size, dtype=torch.long)

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y_pos[idx], self.y_size[idx]


train_loader = DataLoader(
    AcousticDataset(
        X_train_scaled,
        y_pos_train,
        y_size_train
    ),
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    AcousticDataset(
        X_test_scaled,
        y_pos_test,
        y_size_test
    ),
    batch_size=64,
    shuffle=False
)


In [3]:
# =========================
# CNN Model
# =========================
class AcousticCNN(nn.Module):

    def __init__(self, input_length, num_classes):

        super().__init__()

        self.encoder = nn.Sequential(

            nn.Conv1d(1, 64, kernel_size=3, padding=1),
            nn.BatchNorm1d(64),
            nn.GELU(),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.GELU(),

            nn.MaxPool1d(2),

            nn.Conv1d(128, 64, kernel_size=3, padding=1),
            nn.GELU(),

            nn.Flatten()
        )

        # محاسبه داینامیک خروجی Encoder
        with torch.no_grad():

            dummy = torch.zeros(1, 1, input_length)
            output_dim = self.encoder(dummy).shape[1]

        print("Encoder Output Dimension =", output_dim)

        self.fc = nn.Sequential(

            nn.Linear(output_dim, 128),
            nn.GELU(),
            nn.Dropout(0.3)

        )

        self.position_head = nn.Linear(128, 1)

        self.severity_head = nn.Linear(
            128,
            num_classes
        )

    def forward(self, x):

        x = x.unsqueeze(1)

        x = self.encoder(x)

        x = self.fc(x)

        pos = self.position_head(x).squeeze(1)

        size = self.severity_head(x)

        return pos, size



In [4]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = AcousticCNN(
    input_length=input_length,
    num_classes=num_classes
).to(device)

criterion_pos = nn.MSELoss()

criterion_size = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=5
)


Encoder Output Dimension = 704


In [5]:
print(">>> [SYSTEM] Starting Optimized Multi-Task CNN Training...")

epochs = 60

lambda_pos = 1.0
lambda_size = 0.5

for epoch in range(epochs):

    # -------------------
    # Train
    # -------------------
    model.train()

    train_loss = 0

    for bx, byp, bys in train_loader:

        bx = bx.to(device)
        byp = byp.to(device)
        bys = bys.to(device)

        optimizer.zero_grad()

        pred_pos, pred_size = model(bx)

        loss_pos = criterion_pos(pred_pos, byp)

        loss_size = criterion_size(pred_size, bys)

        loss = lambda_pos * loss_pos + lambda_size * loss_size

        loss.backward()

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------------------
    # Validation
    # -------------------
    model.eval()

    val_loss = 0

    with torch.no_grad():

        for bx, byp, bys in test_loader:

            bx = bx.to(device)
            byp = byp.to(device)
            bys = bys.to(device)

            pred_pos, pred_size = model(bx)

            loss = (
                lambda_pos * criterion_pos(pred_pos, byp)
                +
                lambda_size * criterion_size(pred_size, bys)
            )

            val_loss += loss.item()

    val_loss /= len(test_loader)

    scheduler.step(val_loss)

    print(
        f"Epoch {epoch+1:02d}/{epochs} | "
        f"Train Loss = {train_loss:.4f} | "
        f"Val Loss = {val_loss:.4f}"
    )


>>> [SYSTEM] Starting Optimized Multi-Task CNN Training...
Epoch 01/60 | Train Loss = 2002.5498 | Val Loss = 1098.9836
Epoch 02/60 | Train Loss = 1028.7637 | Val Loss = 881.7578
Epoch 03/60 | Train Loss = 876.8197 | Val Loss = 764.7392
Epoch 04/60 | Train Loss = 771.6732 | Val Loss = 668.2538
Epoch 05/60 | Train Loss = 661.9406 | Val Loss = 587.2693
Epoch 06/60 | Train Loss = 616.5886 | Val Loss = 615.1487
Epoch 07/60 | Train Loss = 601.3219 | Val Loss = 573.8801
Epoch 08/60 | Train Loss = 594.6776 | Val Loss = 566.3833
Epoch 09/60 | Train Loss = 595.5385 | Val Loss = 567.7183
Epoch 10/60 | Train Loss = 589.6357 | Val Loss = 563.1517
Epoch 11/60 | Train Loss = 581.8666 | Val Loss = 558.6925
Epoch 12/60 | Train Loss = 587.3566 | Val Loss = 558.0741
Epoch 13/60 | Train Loss = 590.3396 | Val Loss = 557.0407
Epoch 14/60 | Train Loss = 585.8768 | Val Loss = 555.6282
Epoch 15/60 | Train Loss = 578.7736 | Val Loss = 556.6115
Epoch 16/60 | Train Loss = 573.0749 | Val Loss = 556.8814
Epoch 17/6

In [6]:
torch.save(
    model.state_dict(),
    "../models/acoustic_dnn.pth"
)

print(">>> [SUCCESS] Training Complete.")

>>> [SUCCESS] Training Complete.
